In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_parquet('../data/telemetry/silver/cda/Telemetry_Wide_With_States')
mantenciones = pd.read_excel('merge_data.xlsx')
df.head()

,Fecha,Unit,Estado,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,AirFltr,CnkcasePres,...,LtRBrkTemp,Payload,RAftrclrTemp,RtExhTemp,RtFBrkTemp,RtLtExhTemp,RtRBrkTemp,StrgOilTemp,TCOutTemp,TrnLubeTemp
0,2025-01-04 13:18:00,T_12,Operacional,ND,Cargado,-30.253490,-71.091233,1004.105882,3.686355,-0.039465,...,82.368056,193.3,64.053571,452.338160,34.194444,-4.545761,89.127674,68.00000,88.149666,89.963370
1,2025-01-04 00:05:00,T_12,Operacional,Operacional Bajo,Sin Carga,-30.254291,-71.091413,1001.690000,0.940833,0.155318,...,71.600000,0.0,33.696429,239.846830,29.291667,-4.227229,80.000000,68.37500,82.613889,84.351042
2,2025-01-01 23:48:00,T_12,Operacional,Operacional Alto,Sin Carga,NaN,NaN,1032.469091,1.675595,0.175234,...,83.426840,0.0,43.500000,201.080847,31.284722,-2.361279,87.575758,65.42803,87.381239,86.329023
3,2025-01-03 00:14:00,T_12,Operacional,ND,Cargado,-30.246838,-71.099568,944.506897,2.400871,0.287575,...,68.208333,183.8,34.216667,299.669368,26.625000,NaN,76.375000,68.75000,77.633677,81.538091
4,2025-01-05 13:20:00,T_12,Ralenti,Ralenti Alto,Sin Carga,-30.254721,-71.094740,1009.097826,1.412500,0.183102,...,81.274414,0.0,48.255952,157.766434,30.858974,-3.085166,87.166667,68.00000,88.017857,87.989936


In [2]:
mantenciones.rename(columns = {'Fecha_Actual' : 'Fecha', 'Equipos' : 'Unit'}, inplace = True)
mantenciones.Fecha = pd.to_datetime(mantenciones.Fecha, format='%d/%m/%Y')
mantenciones.head(2)

,Unit,Horómetro,Fecha,Hora_de_Detención,Hora_de_Entrega,Tiempo_FS,Nro_Personas,Horas_Persona,Sistema,Sub_Sistemas,Tipo_de_Detención,Trabajo_Ejecutado,Nº_OT,Nº_Backlog,Status_Final,Trabajos_Pendientes,Jefe_de_Turno,Turnos
0,T_24,18134,2026-01-01,07:00,19:00,12,4.0,48,T_Transmision,Diferencial,Programada,en continuación de trabajos asociados a ot:591...,591779,NaN,En Reparacion/Mantencion,puesta en marcha. chequear niveles con equipo ...,Cesar Nilo,Dia
1,T_17,103954,2026-01-01,07:00,19:00,12,2.0,24,T_Mantencion,PM_500,Programada,se recibe equipo detenido por pm de 500 hrs se...,588932,NaN,En Reparacion/Mantencion,"cambiar filtros diferencial, filtros de aceite...",Cesar Nilo,Dia


In [3]:
df_subset = df.loc[:, ['Unit', 'Fecha',
         'EstadoMaquina',
         'EstadoCarga',
         'GPSLat',
         'GPSLon',
         'GPSElevation',
         'GroundSpd',
         'EngSpd',
        "RtRBrkTemp",
        "RtFBrkTemp",
        "LtRBrkTemp",
        "LtFBrkTemp"
]].copy()
df_subset.sort_values(by=['Unit', 'Fecha'], inplace=True)
df_subset.reset_index(drop=True, inplace=True)
df_subset.head()


,Unit,Fecha,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,GroundSpd,EngSpd,RtRBrkTemp,RtFBrkTemp,LtRBrkTemp,LtFBrkTemp
0,T_10,2025-01-01 00:00:00,ND,Sin Carga,-30.254047,-71.090775,998.516666,16.697368,1571.758621,NaN,NaN,NaN,NaN
1,T_10,2025-01-01 00:01:00,ND,Sin Carga,-30.254281,-71.091397,999.499998,3.000000,706.210000,NaN,NaN,NaN,NaN
2,T_10,2025-01-01 00:02:00,ND,Sin Carga,-30.254324,-71.091386,998.488372,0.000000,699.846154,78.833333,80.00,78.0,79.666667
3,T_10,2025-01-01 00:03:00,ND,Sin Carga,-30.254331,-71.091389,996.500000,0.000000,700.000000,78.875000,79.75,78.0,79.500000
4,T_10,2025-01-01 00:08:00,ND,Sin Carga,-30.254323,-71.091415,1001.143333,56.500000,544.300000,NaN,NaN,NaN,NaN


In [4]:
margins = {
        'GPSLat' : (-30.4, -30.1),
        'GPSLon' : (-71.3, -70.9),
        'GPSElevation' : (400, 2000),
        'GroundSpd' : (0, 80),
        'EngSpd' : (0, 2500),
        "RtRBrkTemp" : (20, 200),
        "RtFBrkTemp" : (20, 200),
        "LtRBrkTemp" : (20, 200),
        "LtFBrkTemp" : (20, 200)
}

def clean_data(df_in, margins):
    """Function that uses the margin dict to clean the values -> all the values out of range are replaced by nan"""
    df = df_in.copy()
    for col, (lower, upper) in margins.items():
        df[col] = df[col].where((df[col] >= lower) & (df[col] <= upper), other=pd.NA)
    return df

df_cleaned = clean_data(df_subset, margins)
df_cleaned.dropna(subset=["RtRBrkTemp", "RtFBrkTemp", "LtRBrkTemp", "LtFBrkTemp"], how='all', inplace=True)
df_cleaned.fillna({'EstadoMaquina':'ND', 'EstadoCarga':'Sin Carga'}, inplace=True)
df_cleaned.reset_index(drop=True, inplace=True)
df_cleaned.head()

,Unit,Fecha,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,GroundSpd,EngSpd,RtRBrkTemp,RtFBrkTemp,LtRBrkTemp,LtFBrkTemp
0,T_10,2025-01-01 00:02:00,ND,Sin Carga,-30.254324,-71.091386,998.488372,0.000,699.846154,78.833333,80.000000,78.000000,79.666667
1,T_10,2025-01-01 00:03:00,ND,Sin Carga,-30.254331,-71.091389,996.500000,0.000,700.000000,78.875000,79.750000,78.000000,79.500000
2,T_10,2025-01-01 00:10:00,ND,Sin Carga,-30.254054,-71.091847,1000.935136,7.260,1067.745455,76.666667,76.833333,76.422222,77.500000
3,T_10,2025-01-01 00:11:00,ND,Sin Carga,-30.254104,-71.093598,982.186666,21.200,1824.646552,77.250000,76.416667,77.495238,76.800000
4,T_10,2025-01-01 00:12:00,ND,Sin Carga,-30.252903,-71.096539,939.699995,26.375,1738.591667,79.808824,77.229167,80.878571,76.629545


In [5]:
df_cleaned.isna().sum()

Unit                  0
Fecha                 0
EstadoMaquina         0
EstadoCarga           0
GPSLat            65726
GPSLon            65724
GPSElevation      52310
GroundSpd          5060
EngSpd                6
RtRBrkTemp         4274
RtFBrkTemp       328246
LtRBrkTemp        17829
LtFBrkTemp         5405
dtype: int64

In [6]:
import numpy as np
import pandas as pd

# =========================
# Config
# =========================
UNIT_COL = "Unit"
TIME_COL = "Fecha"
CAT_COLS = ["EstadoMaquina", "EstadoCarga"]  # se mantienen tal cual

FREQ = "1min"
GAP_THRESHOLD = pd.Timedelta("5min")
MIN_DURATION = pd.Timedelta("2h")
MIN_COVERAGE = 0.80
INTERP_LIMIT = 20

df = df_cleaned.copy()

# (opcional, por seguridad)
df = df.drop_duplicates(subset=[UNIT_COL, TIME_COL], keep="last")

# =========================
# 1) Calcular cycle_id en el DF original
# =========================
dt = df.groupby(UNIT_COL)[TIME_COL].diff()
new_cycle = dt.isna() | (dt > GAP_THRESHOLD)
df["cycle_id"] = new_cycle.groupby(df[UNIT_COL]).cumsum().astype("int64")

# =========================
# 2) Funciones auxiliares
# =========================
freq_td = pd.to_timedelta(FREQ)

def is_valid_cycle(cycle_df: pd.DataFrame) -> bool:
    """Ciclo válido si dura >= 2h y tiene suficiente densidad de muestras."""
    start = cycle_df[TIME_COL].iloc[0]
    end = cycle_df[TIME_COL].iloc[-1]
    duration = end - start

    n = len(cycle_df)
    expected_n = int(round(duration / freq_td)) + 1
    coverage = n / expected_n if expected_n > 0 else 0.0

    return (duration >= MIN_DURATION) and (coverage >= MIN_COVERAGE)

def interpolate_cycle(cycle_df: pd.DataFrame) -> pd.DataFrame:
    """Interpola SOLO columnas numéricas dentro del ciclo (time-based)."""
    out = cycle_df.copy()

    # columnas numéricas = señales; excluye unit/time/cycle_id y categóricas
    excluded = {UNIT_COL, TIME_COL, "cycle_id", *CAT_COLS}
    candidates = [c for c in out.columns if c not in excluded]

    # si hay alguna señal que venga como object/string por valores raros, la convertimos
    obj_cols = [c for c in candidates if out[c].dtype == "object" or pd.api.types.is_string_dtype(out[c])]
    for c in obj_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    num_cols = [c for c in candidates if pd.api.types.is_numeric_dtype(out[c])]

    # interpolación
    out = out.set_index(TIME_COL)

    before_na = out[num_cols].isna()
    out[num_cols] = out[num_cols].interpolate(
        method="time",
        limit=INTERP_LIMIT,
        limit_area="inside",
    )
    
    # out[num_cols] = out[num_cols].ffill(limit=3).bfill(limit=3)
    
    out["imputed_any"] = (before_na & ~out[num_cols].isna()).any(axis=1).astype("int8")

    out = out.reset_index()
    return out

# =========================
# 3) Split -> filtrar válidos -> interpolar -> concat
# =========================
from tqdm.auto import tqdm

cycles = []
for (_, _), cycle_df in tqdm(df.groupby([UNIT_COL, "cycle_id"], sort=False), desc="Processing cycles"):

    if is_valid_cycle(cycle_df):
        cycles.append(interpolate_cycle(cycle_df))

cleaned_cycles_df = pd.concat(cycles, ignore_index=True) if cycles else df.head(0).copy()

Processing cycles:   0%|          | 0/24527 [00:00<?, ?it/s]

In [7]:
cleaned_cycles_df.isna().sum()/len(cleaned_cycles_df)*100

Fecha            0.000000
Unit             0.000000
EstadoMaquina    0.000000
EstadoCarga      0.000000
GPSLat           1.246355
GPSLon           1.246355
GPSElevation     1.230283
GroundSpd        0.016182
EngSpd           0.000000
RtRBrkTemp       0.041380
RtFBrkTemp       7.414449
LtRBrkTemp       0.179303
LtFBrkTemp       0.023598
cycle_id         0.000000
imputed_any      0.000000
dtype: float64

In [8]:
# Acceleration = Diff(GroundSpd) / Diff(Fecha) # para cada ciclo, calculamos la aceleración a partir de la velocidad terrestre y el tiempo
cleaned_cycles_df['GroundAcc'] = cleaned_cycles_df.groupby([UNIT_COL, "cycle_id"])['GroundSpd'].diff()
bins = [-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf]
labels = [
    "High Deceleration",
    "Moderate Deceleration",
    "Flat",
    "Moderate Acceleration",
    "High Acceleration",
]

cleaned_cycles_df["GroundAccCategorical"] = pd.cut(
    cleaned_cycles_df["GroundAcc"],
    bins=bins,
    labels=labels,
    right=True,          # (a, b]
    include_lowest=True
)

# Height change = Diff(GPSElevation) # para cada ciclo, calculamos el cambio en la elevación a partir de la diferencia entre muestras consecutivas
cleaned_cycles_df['ElevationChange'] = cleaned_cycles_df.groupby([UNIT_COL, "cycle_id"])['GPSElevation'].diff()
cleaned_cycles_df["ElevationChangeCategorical"] = pd.cut(
    cleaned_cycles_df["ElevationChange"],
    bins=[-np.inf, -10, -5, 5, 10, np.inf],
    labels=["Steep Downhill", "Downhill", "Flat", "Uphill", "Steep Uphill"],
    include_lowest=True
)
# f'{temp}_diff' for temp in brake_temps = [ "RtRBrkTemp", "RtFBrkTemp", "LtRBrkTemp", "LtFBrkTemp" ] # para cada ciclo, calculamos los cambios en la temperatura de los frenos
brake_temps = [ "RtRBrkTemp", "RtFBrkTemp", "LtRBrkTemp", "LtFBrkTemp" ]
for temp in brake_temps:
    cleaned_cycles_df[f'{temp}_diff'] = cleaned_cycles_df.groupby([UNIT_COL, "cycle_id"])[temp].diff()

In [9]:

import statsmodels.formula.api as smf
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

system_prompt = """
You are a data analyst working on a brake degradation study for heavy machinery. You have a cleaned dataset with the following columns:
- payload (categorical): the load state of the machine (e.g., 'Sin Carga', 'Carga Parcial', 'Carga Completa')
- acceleration (numeric): the calculated acceleration based on ground speed
- elevation_change (numeric): the change in elevation based on GPS data
- acc_cat (categorical): binned acceleration category (5 levels)
- elev_cat (categorical): binned elevation change category (5 levels)
- RtRBrkTemp_diff, RtFBrkTemp_diff, LtRBrkTemp_diff, LtFBrkTemp_diff (numeric): the change in brake temperatures for each brake

Your task is to summarize regression findings, focusing on:
- coefficient significance
- R-squared
- AIC/BIC comparison (base vs interaction)
- actionable operational insights for brake degradation / operator feedback.

Regressions being perfored are :
1) Numeric base model: brake_diff ~ C(payload) + acceleration + elevation_change
2) Numeric interaction model: brake_diff ~ C(payload)*acceleration + C(payload)*elevation_change + acceleration*elevation_change
3) Categorical-only model: brake_diff ~ C(payload) + C(acc_cat) + C(elev_cat)
4) Categorical interaction model: brake_diff ~ C(payload)*C(acc_cat) + C(payload)*C(elev_cat) + C(acc_cat)*C(elev_cat)

Dictionary of features:
* RtRBrkTemp_diff: Change in temperature for the Right Rear Brake
* RtFBrkTemp_diff: Change in temperature for the Right Front Brake
* LtRBrkTemp_diff: Change in temperature for the Left Rear Brake
* LtFBrkTemp_diff: Change in temperature for the Left Front Brake

Output:
1. Regression findings : which variables matter most and in what direction? Why?
2. Future Statistical Work : what other analyses would you perform to deepen the understanding of brake degradation? (e.g., non-linear models, time-series analysis, clustering of operational profiles)
3. Future Product/Operational Work : what applications or tools would you build based on these findings? (e.g., real-time operator feedback system, predictive maintenance alerts, driver training modules)
    * Note : This will be a real-world implementation on mining sites. the context is to build a product based on these and future findings by a SaaS company that provides analytics solutions for heavy machinery operations.

Finally, you will suggest actionable insights of applications based on those findings. 
The main goal of the project is to understand the factors that contribute to brake degradation in order to optimize maintenance schedules and improve safety. But also we are interested on understanding the operational conditions that lead to higher brake temperature changes, which can be indicative of more aggressive driving or challenging terrain. This information can be used to provide feedback to operators and potentially adjust operational practices to extend brake life and enhance safety.

"""

# =========================
# 1) Prepare numeric regression dataframe
# =========================
analysis_df_num = cleaned_cycles_df[
    [
        "EstadoCarga",  # payload
        "GroundAcc",
        "ElevationChange",
        "RtRBrkTemp_diff",
        "RtFBrkTemp_diff",
        "LtRBrkTemp_diff",
        "LtFBrkTemp_diff",
    ]
].dropna()

analysis_df_num = analysis_df_num.rename(
    columns={
        "EstadoCarga": "payload",
        "GroundAcc": "acceleration",
        "ElevationChange": "elevation_change",
    }
)

analysis_df_num["payload"] = analysis_df_num["payload"].astype("category")

# =========================
# 2) Prepare categorical regression dataframe
# =========================
analysis_df_cat = cleaned_cycles_df[
    [
        "EstadoCarga",  # payload
        "GroundAccCategorical",
        "ElevationChangeCategorical",
        "RtRBrkTemp_diff",
        "RtFBrkTemp_diff",
        "LtRBrkTemp_diff",
        "LtFBrkTemp_diff",
    ]
].dropna()

analysis_df_cat = analysis_df_cat.rename(
    columns={
        "EstadoCarga": "payload",
        "GroundAccCategorical": "acc_cat",
        "ElevationChangeCategorical": "elev_cat",
    }
)

for c in ["payload", "acc_cat", "elev_cat"]:
    analysis_df_cat[c] = analysis_df_cat[c].astype("category")

# =========================
# 3) Run regression for each brake temperature diff
# =========================
brake_diffs = ["RtRBrkTemp_diff", "RtFBrkTemp_diff", "LtRBrkTemp_diff", "LtFBrkTemp_diff"]

results = {}

for brake in brake_diffs:
    print(f"\n{'='*80}")
    print(f"Analysis for {brake}")
    print(f"{'='*80}")

    # -------------------------
    # NUMERIC MODELS
    # -------------------------
    formula_base = f"{brake} ~ C(payload) + acceleration + elevation_change"
    model_base = smf.ols(formula_base, data=analysis_df_num).fit()

    formula_interact = (
        f"{brake} ~ C(payload) * acceleration + C(payload) * elevation_change + acceleration * elevation_change"
    )
    model_interact = smf.ols(formula_interact, data=analysis_df_num).fit()

    print("\nNUMERIC BASE MODEL:")
    print(model_base.summary())
    print(f"R-squared: {model_base.rsquared:.4f}")
    print(f"AIC: {model_base.aic:.2f} | BIC: {model_base.bic:.2f}")

    print("\nNUMERIC INTERACTION MODEL:")
    print(model_interact.summary())
    print(f"R-squared: {model_interact.rsquared:.4f}")
    print(f"AIC: {model_interact.aic:.2f} | BIC: {model_interact.bic:.2f}")

    # -------------------------
    # CATEGORICAL MODELS
    # -------------------------
    formula_cat = f"{brake} ~ C(payload) + C(acc_cat) + C(elev_cat)"
    model_cat = smf.ols(formula_cat, data=analysis_df_cat).fit()

    formula_cat_inter = (
        f"{brake} ~ C(payload)*C(acc_cat) + C(payload)*C(elev_cat) + C(acc_cat)*C(elev_cat)"
    )
    model_cat_inter = smf.ols(formula_cat_inter, data=analysis_df_cat).fit()

    print("\nCATEGORICAL-ONLY MODEL:")
    print(model_cat.summary())
    print(f"R-squared: {model_cat.rsquared:.4f}")
    print(f"AIC: {model_cat.aic:.2f} | BIC: {model_cat.bic:.2f}")

    print("\nCATEGORICAL INTERACTION MODEL:")
    print(model_cat_inter.summary())
    print(f"R-squared: {model_cat_inter.rsquared:.4f}")
    print(f"AIC: {model_cat_inter.aic:.2f} | BIC: {model_cat_inter.bic:.2f}")

    # -------------------------
    # Simple comparison print
    # -------------------------
    print("\nMODEL COMPARISON (lower is better):")
    print(
        f"NUM base AIC/BIC: {model_base.aic:.2f}/{model_base.bic:.2f} | "
        f"NUM inter AIC/BIC: {model_interact.aic:.2f}/{model_interact.bic:.2f}"
    )
    print(
        f"CAT AIC/BIC: {model_cat.aic:.2f}/{model_cat.bic:.2f} | "
        f"CAT inter AIC/BIC: {model_cat_inter.aic:.2f}/{model_cat_inter.bic:.2f}"
    )

    # -------------------------
    # Group summaries (actionable)
    # -------------------------
    means_table = (
        analysis_df_cat.groupby(["payload", "acc_cat", "elev_cat"], observed=True)[brake]
        .agg(["count", "mean", "std"])
        .sort_values("mean", ascending=False)
        .head(15)
    )

    print("\nTOP 15 GROUP MEANS (payload x acc_cat x elev_cat):")
    print(means_table)

    # -------------------------
    # LLM insights (optional)
    # -------------------------
    history = [
        {"role": "developer", "content": system_prompt},
        {
            "role": "user",
            "content": f"""
Here are the regression outputs for {brake}.

NUMERIC BASE MODEL:
{model_base.summary()}

NUMERIC INTERACTION MODEL:
{model_interact.summary()}

CATEGORICAL-ONLY MODEL:
{model_cat.summary()}

CATEGORICAL INTERACTION MODEL:
{model_cat_inter.summary()}

R-squared:
- Numeric base: {model_base.rsquared:.4f}
- Numeric interaction: {model_interact.rsquared:.4f}
- Categorical only: {model_cat.rsquared:.4f}
- Categorical interaction: {model_cat_inter.rsquared:.4f}

AIC / BIC:
- Numeric base: AIC={model_base.aic:.2f}, BIC={model_base.bic:.2f}
- Numeric interaction: AIC={model_interact.aic:.2f}, BIC={model_interact.bic:.2f}
- Categorical only: AIC={model_cat.aic:.2f}, BIC={model_cat.bic:.2f}
- Categorical interaction: AIC={model_cat_inter.aic:.2f}, BIC={model_cat_inter.bic:.2f}

Top group means (payload x acc_cat x elev_cat):
{means_table.to_string()}

Please summarize:
1) Which variables matter most and in what direction?
2) Whether interaction terms improve fit (use AIC/BIC + R²).
3) Operational takeaways (operator feedback, terrain, payload guidance).
""",
        },
    ]

    response = client.responses.create(
        model="gpt-5-mini",
        input=history,
    )

    print("\nLLM INSIGHTS:")
    print(response.output_text)

    results[brake] = {
        "numeric_base": model_base,
        "numeric_interaction": model_interact,
        "categorical": model_cat,
        "categorical_interaction": model_cat_inter,
        "top_group_means": means_table,
        "insights": response.output_text,
    }
    
import json
with open("regression_results.json", "w") as f:
    json.dump(results, f, default=str)


Analysis for RtRBrkTemp_diff

NUMERIC BASE MODEL:
                            OLS Regression Results                            
Dep. Variable:        RtRBrkTemp_diff   R-squared:                       0.221
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                 1.870e+05
Date:                Wed, 04 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:53:24   Log-Likelihood:            -5.6933e+06
No. Observations:             3298043   AIC:                         1.139e+07
Df Residuals:                 3298037   BIC:                         1.139e+07
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------